# 🏍️ Two-Wheeler Loan Risk Analytics: EDA & Data Cleaning
### Credit Risk Intelligence for NBFC Two-Wheeler Portfolio
**Objective:** Ingest 20,000 borrower records, detect missingness and anomalies, clean records via median imputation and winsorization, inspect class imbalance, and extract key underwriting insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded successfully.")

## 1. Data Ingestion
Loading raw two-wheeler loan portfolio data from `./data/two_wheeler_loans.csv`.

In [ ]:
df = pd.read_csv('../data/two_wheeler_loans.csv')
print(f"Dataset Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

## 2. Missing Value Analysis & Strategy
Checking column-wise missing percentages. Deliberate missingness injected at ~4% across financial attributes.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
missing_table[missing_table['Missing Count'] > 0]

In [ ]:
# Imputation Strategy: Use median imputation to preserve sample size and prevent loss of rare default cases
for col in ['monthly_income', 'credit_score', 'down_payment_pct', 'interest_rate']:
    df[col] = df[col].fillna(df[col].median())
print("Missing values after imputation:", df.isnull().sum().sum())

## 3. Outlier Detection & Capping (Winsorization)
Inspect continuous distributions for unrealistic extremes (e.g. synthetic anomalies).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(y=df['monthly_income'], ax=axes[0], color='#3498db').set_title('Monthly Income (Raw)')
sns.boxplot(y=df['loan_amount'], ax=axes[1], color='#e67e22').set_title('Loan Amount (Raw)')
sns.boxplot(y=df['credit_score'], ax=axes[2], color='#2ecc71').set_title('Credit Score (Raw)')
plt.tight_layout()
plt.show()

In [ ]:
# Winsorization: Cap numeric outliers at 1st - 99th percentiles & drop invalid credit scores > 900
for col in ['monthly_income', 'loan_amount', 'emi']:
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(lo, hi)

df = df[df['credit_score'] <= 900].copy()
print(f"Cleaned records count: {len(df)}")

## 4. Class Imbalance Analysis
Evaluating positive rate of the target `defaulted` (0 = Non-Default, 1 = Default).

In [ ]:
default_dist = df['defaulted'].value_counts(normalize=True) * 100
print(f"Overall Default Rate: {default_dist[1]:.2f}%")

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='defaulted', palette=['#2ecc71', '#e74c3c'])
plt.title(f"Target Distribution (Default Rate: {default_dist[1]:.2f}%)")
plt.xlabel("Default Status (0 = Repaid, 1 = Defaulted)")
plt.ylabel("Number of Borrowers")
plt.show()

> **Insight:** The default rate is ~11.36%, indicating moderate class imbalance (~9:1). Standard accuracy would be misleading (~89% baseline accuracy by predicting zero). We must evaluate with ROC-AUC, PR-AUC, and tune decision thresholds.

## 5. Feature Engineering
Deriving essential NBFC credit risk metrics: EMI-to-income, LTV ratio, and behavioural flags.

In [ ]:
# 1. EMI-to-income (FOIR)
df['emi_to_income'] = (df['emi'] / df['monthly_income'].clip(1)).round(4)

# 2. Loan-to-Value (LTV)
df['ltv_ratio'] = (df['loan_amount'] / df['vehicle_price'].clip(1)).round(4)

# 3. Categorical risk bands
df['income_band'] = pd.cut(df['monthly_income'], bins=[0, 15000, 25000, 40000, 60000, np.inf],
                           labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
df['credit_band'] = pd.cut(df['credit_score'], bins=[0, 580, 650, 720, 790, 900],
                           labels=['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'])

# 4. Behavioural flags
df['late_pmt_flag'] = (df['late_payments_last_12m'] > 0).astype(int)
df['high_emi_burden'] = (df['emi_to_income'] > 0.40).astype(int)

# 5. Temporal
df['loan_year'] = pd.to_datetime(df['loan_start_date']).dt.year
df['loan_month'] = pd.to_datetime(df['loan_start_date']).dt.month

df[['emi_to_income', 'ltv_ratio', 'late_pmt_flag', 'high_emi_burden']].head()

## 6. Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, cbar=True)
plt.title("Correlation Matrix of Financial & Risk Attributes")
plt.show()

> **Insight:** Late payments (`late_payments_last_12m`) and `emi_to_income` exhibit strong positive correlation with default, while `credit_score` is strongly inversely correlated. Cleaned data is ready for modeling.